# Práctica de A2C (Advantage Actor-Critic)

Este cuaderno implementa **A2C** desde cero en PyTorch (con *n-step returns*, GAE opcional y varios entornos en paralelo) para que puedas **experimentar con los hiperparámetros** desde un formulario, sin tocar el código.

**Cómo usarlo**
1. Ejecutá la celda de instalación (una sola vez por sesión) y la de imports.
2. Cambiá los valores en la celda **"Hiperparámetros"** (sliders y desplegables con rangos mínimo/máximo).
3. Ejecutá la validación, la definición del agente y el entrenamiento.
4. Mirá las curvas, la evaluación y (opcional) el video.

**Ruta de entornos sugerida** (de más fácil a más difícil para A2C):

| Etapa | Entorno | Acciones | Para qué sirve |
|---|---|---|---|
| 1 | `CartPole-v1` | discretas | Ver que A2C aprende en minutos; probar `n_steps`, `gamma`, `lr` |
| 2 | `Acrobot-v1` | discretas | Recompensas dispersas/negativas; efecto de `gamma` y `GAE λ` |
| 3 | `LunarLander-v3` | discretas | Entorno "serio" discreto; efecto de entropía y `n_envs` |
| 4 | `Pendulum-v1` | continuas | Primer paso a acciones continuas (política gaussiana) |
| 5 | `Hopper-v5` | continuas | MuJoCo con terminación temprana |
| 6 | `HalfCheetah-v5` | continuas | Sin caídas: ideal para aislar el efecto de los hiperparámetros |
| 7 | `Walker2d-v5` | continuas | Más inestable; importa normalizar observaciones |
| 8 | `BipedalWalker-v3` | continuas | El más difícil: recompensas ruidosas, requiere muchos pasos |

## 1. Instalación (ejecutar una vez)

In [ ]:
%%capture
!apt-get install -y swig
!pip install -q "gymnasium[box2d,mujoco]" imageio imageio-ffmpeg

## 2. Imports

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"   # render sin pantalla (para el video)

import time, base64, random
import numpy as np
import torch, torch.nn as nn
from torch.distributions import Categorical, Normal
import gymnasium as gym
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device, "| gymnasium", gym.__version__)

## 3. Hiperparámetros (formulario)

In [ ]:
#@title Hiperparámetros de A2C { display-mode: "form" }

#@markdown ### Entorno y ejecución
ENV_ID = "CartPole-v1" #@param ["CartPole-v1", "Acrobot-v1", "LunarLander-v3", "Pendulum-v1", "Hopper-v5", "HalfCheetah-v5", "Walker2d-v5", "BipedalWalker-v3"]
SEED = 1 #@param {type:"slider", min:0, max:100, step:1}
TOTAL_TIMESTEPS = 200000 #@param {type:"slider", min:50000, max:2000000, step:50000}

#@markdown ### Recolección de experiencia
N_ENVS = 8 #@param {type:"slider", min:1, max:16, step:1}
N_STEPS = 8 #@param {type:"slider", min:1, max:128, step:1}

#@markdown ### Retornos y ventajas
GAMMA = 0.99 #@param {type:"slider", min:0.90, max:0.9999, step:0.0001}
GAE_LAMBDA = 1.0 #@param {type:"slider", min:0.80, max:1.0, step:0.01}
NORMALIZE_ADVANTAGE = False #@param {type:"boolean"}

#@markdown ### Optimización
OPTIMIZER = "RMSprop" #@param ["RMSprop", "Adam"]
LEARNING_RATE = 0.0007 #@param {type:"slider", min:0.00001, max:0.003, step:0.00001}
LR_DECAY = True #@param {type:"boolean"}
ENT_COEF = 0.0 #@param {type:"slider", min:0.0, max:0.1, step:0.001}
VF_COEF = 0.5 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
MAX_GRAD_NORM = 0.5 #@param {type:"slider", min:0.1, max:5.0, step:0.1}

#@markdown ### Red neuronal
HIDDEN_SIZE = 64 #@param {type:"slider", min:16, max:256, step:16}
HIDDEN_LAYERS = 2 #@param {type:"slider", min:1, max:3, step:1}
LOG_STD_INIT = 0.0 #@param {type:"slider", min:-2.0, max:0.5, step:0.1}

#@markdown ### Preprocesamiento
NORMALIZE_OBS = True #@param {type:"boolean"}
REWARD_SCALE = 1.0 #@param {type:"slider", min:0.01, max:1.0, step:0.01}

#@markdown ### Evaluación
EVAL_EPISODES = 5 #@param {type:"slider", min:1, max:20, step:1}
GRABAR_VIDEO = False #@param {type:"boolean"}
VIDEO_MAX_PASOS = 500 #@param {type:"slider", min:200, max:5000, step:100}
VIDEO_FPS = 30 #@param {type:"slider", min:5, max:60, step:5}

## 4. Validación de rangos

In [ ]:
RANGES = {
    "SEED": (0, 100), "TOTAL_TIMESTEPS": (50_000, 2_000_000),
    "N_ENVS": (1, 16), "N_STEPS": (1, 128),
    "GAMMA": (0.90, 0.9999), "GAE_LAMBDA": (0.80, 1.0),
    "LEARNING_RATE": (1e-5, 3e-3), "ENT_COEF": (0.0, 0.1),
    "VF_COEF": (0.1, 1.0), "MAX_GRAD_NORM": (0.1, 5.0),
    "HIDDEN_SIZE": (16, 256), "HIDDEN_LAYERS": (1, 3),
    "LOG_STD_INIT": (-2.0, 0.5), "REWARD_SCALE": (0.01, 1.0),
    "EVAL_EPISODES": (1, 20),
    "VIDEO_MAX_PASOS": (200, 5000), "VIDEO_FPS": (5, 60),
}
INT_PARAMS = {"SEED", "TOTAL_TIMESTEPS", "N_ENVS", "N_STEPS", "HIDDEN_SIZE", "HIDDEN_LAYERS", "EVAL_EPISODES",
              "VIDEO_MAX_PASOS", "VIDEO_FPS"}

for name, (lo, hi) in RANGES.items():
    v = globals()[name]
    c = min(max(v, lo), hi)
    if name in INT_PARAMS:
        c = int(c)
    if c != v:
        print(f"Aviso: {name}={v} está fuera de [{lo}, {hi}]; se usará {c}")
        globals()[name] = c

SUGERENCIAS = {
    "CartPole-v1":      "n_envs=8, n_steps=5-8, lr=7e-4, 100k-200k pasos",
    "Acrobot-v1":       "n_envs=8, n_steps=8, gamma=0.99, 200k-500k pasos",
    "LunarLander-v3":   "n_envs=8, n_steps=16-32, lr=7e-4, ent_coef=0.01, 500k-1M pasos",
    "Pendulum-v1":      "n_envs=8, n_steps=8-16, normalizar obs, reward_scale~0.1, 500k pasos",
    "Hopper-v5":        "n_envs=4-8, n_steps=8-16, normalizar obs, Adam lr~3e-4, 1M pasos",
    "HalfCheetah-v5":   "n_envs=4-8, n_steps=8-16, normalizar obs, Adam lr~3e-4, 1M-2M pasos",
    "Walker2d-v5":      "n_envs=8, n_steps=16, normalizar obs, Adam lr~3e-4, 1M-2M pasos",
    "BipedalWalker-v3": "n_envs=8-16, n_steps=16-32, normalizar obs, reward_scale~0.1, 2M pasos",
}
print("Entorno:", ENV_ID)
print("Sugerencia:", SUGERENCIAS[ENV_ID])
print("Batch por actualización:", N_ENVS * N_STEPS, "transiciones")

## 5. Agente A2C

In [ ]:
class RunningMeanStd:
    def __init__(self, shape):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = 1e-4

    def update(self, x):
        x = np.asarray(x, dtype=np.float64)
        bm, bv, bc = x.mean(axis=0), x.var(axis=0), x.shape[0]
        delta = bm - self.mean
        tot = self.count + bc
        new_mean = self.mean + delta * bc / tot
        m2 = self.var * self.count + bv * bc + delta**2 * self.count * bc / tot
        self.mean, self.var, self.count = new_mean, m2 / tot, tot

    def normalize(self, x):
        return np.clip((x - self.mean) / np.sqrt(self.var + 1e-8), -10, 10)


def mlp(inp, out, hidden, layers, out_gain):
    mods, d = [], inp
    for _ in range(layers):
        lin = nn.Linear(d, hidden)
        nn.init.orthogonal_(lin.weight, np.sqrt(2)); nn.init.zeros_(lin.bias)
        mods += [lin, nn.Tanh()]
        d = hidden
    last = nn.Linear(d, out)
    nn.init.orthogonal_(last.weight, out_gain); nn.init.zeros_(last.bias)
    mods.append(last)
    return nn.Sequential(*mods)


class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, discrete, hidden, layers, log_std_init):
        super().__init__()
        self.discrete = discrete
        self.actor = mlp(obs_dim, act_dim, hidden, layers, 0.01)
        self.critic = mlp(obs_dim, 1, hidden, layers, 1.0)
        if not discrete:
            self.log_std = nn.Parameter(torch.full((act_dim,), float(log_std_init)))

    def dist(self, x):
        out = self.actor(x)
        if self.discrete:
            return Categorical(logits=out)
        return Normal(out, self.log_std.exp().expand_as(out))

    def value(self, x):
        return self.critic(x).squeeze(-1)

    def act(self, x, deterministic=False):
        d = self.dist(x)
        if deterministic:
            a = d.logits.argmax(-1) if self.discrete else d.mean
        else:
            a = d.sample()
        return a

    def evaluate(self, x, a):
        d = self.dist(x)
        logp = d.log_prob(a)
        ent = d.entropy()
        if not self.discrete:
            logp, ent = logp.sum(-1), ent.sum(-1)
        return logp, ent, self.value(x)


def make_env(env_id, **kw):
    return gym.make(env_id, **kw)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

## 6. Entrenamiento

In [ ]:
set_seed(SEED)
envs = [make_env(ENV_ID) for _ in range(N_ENVS)]
obs_space, act_space = envs[0].observation_space, envs[0].action_space
discrete = isinstance(act_space, gym.spaces.Discrete)
obs_dim = int(np.prod(obs_space.shape))
act_dim = int(act_space.n) if discrete else int(np.prod(act_space.shape))
if not discrete:
    a_low, a_high = act_space.low, act_space.high

model = ActorCritic(obs_dim, act_dim, discrete, HIDDEN_SIZE, HIDDEN_LAYERS, LOG_STD_INIT).to(device)
if OPTIMIZER == "RMSprop":
    opt = torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE, alpha=0.99, eps=1e-5)
else:
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, eps=1e-5)

rms = RunningMeanStd((obs_dim,))

def prep(raw, update=False):
    raw = np.asarray(raw, dtype=np.float64).reshape(-1, obs_dim)
    if NORMALIZE_OBS:
        if update:
            rms.update(raw)
        raw = rms.normalize(raw)
    return torch.as_tensor(raw, dtype=torch.float32, device=device)

def to_env_action(a):
    a = a.cpu().numpy()
    if discrete:
        return [int(x) for x in a]
    return [np.clip(x, a_low, a_high) for x in a]

obs = np.stack([e.reset(seed=SEED + i)[0] for i, e in enumerate(envs)])
ep_ret = np.zeros(N_ENVS); ep_len = np.zeros(N_ENVS, dtype=int)
ret_hist, step_hist = [], []
log = {"upd_step": [], "vloss": [], "ploss": [], "entropy": []}

n_updates = TOTAL_TIMESTEPS // (N_ENVS * N_STEPS)
global_step = 0
t0 = time.time()

for upd in range(n_updates):
    if LR_DECAY:
        for g in opt.param_groups:
            g["lr"] = LEARNING_RATE * (1 - upd / n_updates)

    b_obs = torch.zeros((N_STEPS, N_ENVS, obs_dim), device=device)
    b_act = torch.zeros((N_STEPS, N_ENVS) + (() if discrete else (act_dim,)),
                        device=device, dtype=torch.long if discrete else torch.float32)
    b_rew = torch.zeros((N_STEPS, N_ENVS), device=device)
    b_done = torch.zeros((N_STEPS, N_ENVS), device=device)
    b_val = torch.zeros((N_STEPS, N_ENVS), device=device)

    for t in range(N_STEPS):
        x = prep(obs, update=True)
        with torch.no_grad():
            a = model.act(x)
            v = model.value(x)
        b_obs[t], b_act[t], b_val[t] = x, a, v

        new_obs = []
        for i, (e, ea) in enumerate(zip(envs, to_env_action(a))):
            o, r, term, trunc, _ = e.step(ea)
            ep_ret[i] += r; ep_len[i] += 1
            r_train = r * REWARD_SCALE
            if trunc and not term:
                # corte por tiempo: no es un estado terminal real, se hace bootstrap
                with torch.no_grad():
                    vb = model.value(prep(o)).item()
                r_train += GAMMA * vb
            b_rew[t, i] = r_train
            if term or trunc:
                b_done[t, i] = 1.0
                ret_hist.append(ep_ret[i]); step_hist.append(global_step + (t + 1) * N_ENVS)
                ep_ret[i] = 0; ep_len[i] = 0
                o, _ = e.reset()
            new_obs.append(o)
        obs = np.stack(new_obs)

    global_step += N_ENVS * N_STEPS

    with torch.no_grad():
        last_v = model.value(prep(obs))
    adv = torch.zeros_like(b_rew)
    last = torch.zeros(N_ENVS, device=device)
    for t in reversed(range(N_STEPS)):
        nv = last_v if t == N_STEPS - 1 else b_val[t + 1]
        nonterm = 1.0 - b_done[t]
        delta = b_rew[t] + GAMMA * nv * nonterm - b_val[t]
        last = delta + GAMMA * GAE_LAMBDA * nonterm * last
        adv[t] = last
    returns = adv + b_val

    f_obs = b_obs.reshape(-1, obs_dim)
    f_act = b_act.reshape((-1,) + b_act.shape[2:])
    f_adv, f_ret = adv.reshape(-1), returns.reshape(-1)
    if NORMALIZE_ADVANTAGE and f_adv.numel() > 1:
        f_adv = (f_adv - f_adv.mean()) / (f_adv.std() + 1e-8)

    logp, ent, val = model.evaluate(f_obs, f_act)
    p_loss = -(logp * f_adv).mean()
    v_loss = ((f_ret - val) ** 2).mean()
    loss = p_loss + VF_COEF * v_loss - ENT_COEF * ent.mean()

    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    opt.step()

    log["upd_step"].append(global_step)
    log["vloss"].append(v_loss.item()); log["ploss"].append(p_loss.item())
    log["entropy"].append(ent.mean().item())

    if (upd + 1) % max(1, n_updates // 20) == 0:
        rec = np.mean(ret_hist[-20:]) if ret_hist else float("nan")
        print(f"paso {global_step:>8d} | retorno medio (últ. 20 ep.): {rec:8.1f} | "
              f"entropía {log['entropy'][-1]:.3f} | {time.time()-t0:.0f}s")

print("Entrenamiento terminado.")

## 7. Curvas de aprendizaje

In [ ]:
def smooth(y, k=20):
    y = np.asarray(y, dtype=float)
    if len(y) < k:
        return y
    return np.convolve(y, np.ones(k) / k, mode="valid")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

if ret_hist:
    k = min(20, len(ret_hist))
    ax[0].plot(step_hist, ret_hist, alpha=0.25, label="episodio")
    ax[0].plot(step_hist[k-1:], smooth(ret_hist, k), label=f"media móvil ({k})")
ax[0].set_title("Retorno por episodio"); ax[0].set_xlabel("pasos"); ax[0].legend()

k2 = max(1, min(10, len(log["vloss"])))
ax[1].plot(log["upd_step"][k2-1:], smooth(log["vloss"], k2), label="value loss")
ax[1].set_yscale("log"); ax[1].set_title("Pérdida del crítico"); ax[1].set_xlabel("pasos")

ax[2].plot(log["upd_step"], log["entropy"])
ax[2].set_title("Entropía de la política"); ax[2].set_xlabel("pasos")
plt.tight_layout(); plt.show()

## 8. Evaluación (política determinista) y video opcional

In [ ]:
def evaluar(n_ep):
    env = make_env(ENV_ID)
    rets = []
    for ep in range(n_ep):
        o, _ = env.reset(seed=10_000 + ep)
        done, R = False, 0.0
        while not done:
            with torch.no_grad():
                a = model.act(prep(o), deterministic=True)
            o, r, term, trunc, _ = env.step(to_env_action(a)[0])
            R += r; done = term or trunc
        rets.append(R)
    env.close()
    return np.array(rets)

rets = evaluar(EVAL_EPISODES)
print(f"Evaluación en {EVAL_EPISODES} episodios: {rets.mean():.1f} ± {rets.std():.1f}")

if GRABAR_VIDEO:
    try:
        import imageio
        from IPython.display import HTML, display
        # max_episode_steps reemplaza el límite de tiempo propio del entorno (p. ej. 500 en Acrobot)
        env = make_env(ENV_ID, render_mode="rgb_array", max_episode_steps=VIDEO_MAX_PASOS)
        o, _ = env.reset(seed=123)
        frames, done = [], False
        while not done and len(frames) < VIDEO_MAX_PASOS:
            frames.append(env.render())
            with torch.no_grad():
                a = model.act(prep(o), deterministic=True)
            o, r, term, trunc, _ = env.step(to_env_action(a)[0])
            done = term or trunc
        env.close()
        imageio.mimsave("episodio.mp4", frames, fps=VIDEO_FPS)
        b64 = base64.b64encode(open("episodio.mp4", "rb").read()).decode()
        display(HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
    except Exception as e:
        print("No se pudo grabar el video:", e)

## 9. Ejercicios sugeridos

1. **`n_steps` y varianza**: con `CartPole-v1`, compará `n_steps` = 1, 5, 32 con el mismo `seed`. ¿Qué pasa con la estabilidad y la velocidad?
2. **`GAE λ`**: con `Acrobot-v1`, probá λ = 0.8, 0.95, 1.0. Relacionalo con el compromiso sesgo/varianza.
3. **`n_envs`**: manteniendo `n_envs × n_steps` constante, variá la distribución entre ambos. ¿Cambia algo?
4. **Entropía**: en `LunarLander-v3`, comparado `ent_coef` = 0, 0.01, 0.05. Mirá la curva de entropía.
5. **Normalización de observaciones**: en `Hopper-v5` o `Walker2d-v5`, apagá `NORMALIZE_OBS` y comparalo con encendido.
6. **Comparar con REINFORCE con baseline**: en `HalfCheetah-v5` y `Walker2d-v5`, ¿qué ventaja aporta el crítico con bootstrap sobre el retorno de Monte Carlo completo?
7. **Varias semillas**: repetí con 3 `seed` distintos y reportá media y desvío; A2C es muy sensible a la semilla.